
# Assignment 7 — Creating and Fixing Mode Collapse in a GAN

## Objective
This notebook demonstrates **mode collapse** in a DCGAN and then applies **label smoothing** to improve training stability.

The experiment follows the assignment requirements:
- Start with a DCGAN-style generator and discriminator.
- Deliberately destabilize the model.
- Train the broken version for approximately 15–20 epochs.
- Save/display generated images showing the collapsed output.
- Apply label smoothing.
- Retrain and compare the improved output.
- Answer the four required written questions at the end.


## 1. Import Libraries

In [ ]:

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

from torchvision.utils import make_grid, save_image
from torch.utils.data import DataLoader

print("PyTorch version:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("assignment7_outputs", exist_ok=True)



## 2. Load the Dataset

Fashion-MNIST is used because it is small and easy to train with a DCGAN. The images are converted to tensors and normalized to the range `[-1, 1]`, which matches the generator's `Tanh` output.


In [ ]:

BATCH_SIZE = 128
IMAGE_SIZE = 64

transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = torchvision.datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

print("Training images:", len(dataset))


## 3. Define the DCGAN Models

In [ ]:

class Generator(nn.Module):
    def __init__(self, latent_dim=100, base_channels=64, use_batchnorm=True):
        super().__init__()

        layers = [
            nn.ConvTranspose2d(latent_dim, base_channels * 8, 4, 1, 0, bias=False)
        ]

        if use_batchnorm:
            layers.append(nn.BatchNorm2d(base_channels * 8))

        layers += [
            nn.ReLU(True),
            nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 4, 2, 1, bias=False)
        ]

        if use_batchnorm:
            layers.append(nn.BatchNorm2d(base_channels * 4))

        layers += [
            nn.ReLU(True),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 4, 2, 1, bias=False)
        ]

        if use_batchnorm:
            layers.append(nn.BatchNorm2d(base_channels * 2))

        layers += [
            nn.ReLU(True),
            nn.ConvTranspose2d(base_channels * 2, base_channels, 4, 2, 1, bias=False)
        ]

        if use_batchnorm:
            layers.append(nn.BatchNorm2d(base_channels))

        layers += [
            nn.ReLU(True),
            nn.ConvTranspose2d(base_channels, 1, 4, 2, 1, bias=False),
            nn.Tanh()
        ]

        self.model = nn.Sequential(*layers)

    def forward(self, z):
        return self.model(z)


class Discriminator(nn.Module):
    def __init__(self, base_channels=64):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(1, base_channels, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_channels * 2, base_channels * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_channels * 4, base_channels * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels * 8),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_channels * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x).view(-1)



## 4. Helper Functions

The following functions initialize the models, generate a fixed set of images, and display/save image grids. Using the same fixed noise before and after training makes the visual comparison easier.


In [ ]:

LATENT_DIM = 100
NUM_FIXED_IMAGES = 64

torch.manual_seed(42)
fixed_noise = torch.randn(NUM_FIXED_IMAGES, LATENT_DIM, 1, 1, device=device)

def initialize_weights(model):
    for layer in model.modules():
        if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(layer.weight.data, 0.0, 0.02)
        elif isinstance(layer, nn.BatchNorm2d):
            nn.init.normal_(layer.weight.data, 1.0, 0.02)
            nn.init.constant_(layer.bias.data, 0)

def show_generated_images(generator, title, filename):
    generator.eval()
    with torch.no_grad():
        images = generator(fixed_noise).cpu()

    grid = make_grid(images, nrow=8, normalize=True, value_range=(-1, 1))
    save_image(grid, filename)

    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

    generator.train()



# Part A — Deliberately Create Mode Collapse

## 5. Create an Unstable Version of the DCGAN

To make the model unstable, this version uses **two modifications** allowed by the assignment:
1. The generator capacity is reduced.
2. Batch normalization is removed from the generator.

A relatively high learning rate is also used to make the adversarial game harder to stabilize.

The goal is not to create a useful final model here; the goal is to observe whether the generator starts producing very similar outputs.


In [ ]:

# Deliberately unstable settings
BROKEN_EPOCHS = 15
BROKEN_G_CHANNELS = 16
BROKEN_LR = 0.001

broken_G = Generator(
    latent_dim=LATENT_DIM,
    base_channels=BROKEN_G_CHANNELS,
    use_batchnorm=False
).to(device)

broken_D = Discriminator(
    base_channels=64
).to(device)

initialize_weights(broken_G)
initialize_weights(broken_D)

criterion = nn.BCELoss()

optimizer_G_broken = optim.Adam(
    broken_G.parameters(),
    lr=BROKEN_LR,
    betas=(0.5, 0.999)
)

optimizer_D_broken = optim.Adam(
    broken_D.parameters(),
    lr=BROKEN_LR,
    betas=(0.5, 0.999)
)

print("Broken generator parameters:",
      sum(p.numel() for p in broken_G.parameters()))
print("Broken discriminator parameters:",
      sum(p.numel() for p in broken_D.parameters()))


## 6. Train the Unstable Model

In [ ]:

broken_G.train()
broken_D.train()

broken_g_losses = []
broken_d_losses = []

for epoch in range(BROKEN_EPOCHS):
    running_g = 0.0
    running_d = 0.0

    for real_images, _ in dataloader:
        real_images = real_images.to(device)
        batch_size = real_images.size(0)

        # -------------------------
        # Train Discriminator
        # -------------------------
        optimizer_D_broken.zero_grad()

        real_labels = torch.ones(batch_size, device=device)
        fake_labels = torch.zeros(batch_size, device=device)

        real_output = broken_D(real_images)
        loss_real = criterion(real_output, real_labels)

        noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
        fake_images = broken_G(noise)

        fake_output = broken_D(fake_images.detach())
        loss_fake = criterion(fake_output, fake_labels)

        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D_broken.step()

        # -------------------------
        # Train Generator
        # -------------------------
        optimizer_G_broken.zero_grad()

        fake_output = broken_D(fake_images)
        loss_G = criterion(fake_output, real_labels)

        loss_G.backward()
        optimizer_G_broken.step()

        running_g += loss_G.item()
        running_d += loss_D.item()

    avg_g = running_g / len(dataloader)
    avg_d = running_d / len(dataloader)

    broken_g_losses.append(avg_g)
    broken_d_losses.append(avg_d)

    print(
        f"Epoch [{epoch + 1}/{BROKEN_EPOCHS}] "
        f"Generator Loss: {avg_g:.4f} | "
        f"Discriminator Loss: {avg_d:.4f}"
    )



## 7. Observe the Collapsed Output

If mode collapse occurs, many generated images will have very similar shapes, textures, or overall appearance instead of covering different modes of the training distribution.

**Take a screenshot of the output below for submission.**


In [ ]:

show_generated_images(
    broken_G,
    "Broken DCGAN - Mode Collapse Output",
    "assignment7_outputs/broken_mode_collapse.png"
)


In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(broken_g_losses, label="Generator Loss")
plt.plot(broken_d_losses, label="Discriminator Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Broken DCGAN Training Loss")
plt.legend()
plt.grid(True)
plt.show()



# Part B — Fix the Problem

## 8. Apply Label Smoothing

The improvement technique used here is **label smoothing**.

Instead of training the discriminator with:
- real = `1.0`
- fake = `0.0`

we use:
- real = `0.9`
- fake = `0.0`

This prevents the discriminator from becoming excessively confident and can make the adversarial training process less sharp and more stable.


In [ ]:

FIXED_EPOCHS = 15

# Use a more stable generator configuration for the retraining experiment.
fixed_G = Generator(
    latent_dim=LATENT_DIM,
    base_channels=64,
    use_batchnorm=True
).to(device)

fixed_D = Discriminator(
    base_channels=64
).to(device)

initialize_weights(fixed_G)
initialize_weights(fixed_D)

# Slightly lower learning rates for the stabilized training.
FIXED_LR_G = 0.0002
FIXED_LR_D = 0.0002

optimizer_G_fixed = optim.Adam(
    fixed_G.parameters(),
    lr=FIXED_LR_G,
    betas=(0.5, 0.999)
)

optimizer_D_fixed = optim.Adam(
    fixed_D.parameters(),
    lr=FIXED_LR_D,
    betas=(0.5, 0.999)
)

SMOOTH_REAL = 0.9

print("Label smoothing value for real images:", SMOOTH_REAL)


## 9. Retrain the Improved DCGAN

In [ ]:

fixed_G.train()
fixed_D.train()

fixed_g_losses = []
fixed_d_losses = []

for epoch in range(FIXED_EPOCHS):
    running_g = 0.0
    running_d = 0.0

    for real_images, _ in dataloader:
        real_images = real_images.to(device)
        batch_size = real_images.size(0)

        # -------------------------
        # Train Discriminator
        # -------------------------
        optimizer_D_fixed.zero_grad()

        # Label smoothing: real labels are 0.9 instead of 1.0
        real_labels = torch.full(
            (batch_size,),
            SMOOTH_REAL,
            device=device
        )

        fake_labels = torch.zeros(batch_size, device=device)

        real_output = fixed_D(real_images)
        loss_real = criterion(real_output, real_labels)

        noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
        fake_images = fixed_G(noise)

        fake_output = fixed_D(fake_images.detach())
        loss_fake = criterion(fake_output, fake_labels)

        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizer_D_fixed.step()

        # -------------------------
        # Train Generator
        # -------------------------
        optimizer_G_fixed.zero_grad()

        fake_output = fixed_D(fake_images)
        generator_targets = torch.ones(batch_size, device=device)

        loss_G = criterion(fake_output, generator_targets)

        loss_G.backward()
        optimizer_G_fixed.step()

        running_g += loss_G.item()
        running_d += loss_D.item()

    avg_g = running_g / len(dataloader)
    avg_d = running_d / len(dataloader)

    fixed_g_losses.append(avg_g)
    fixed_d_losses.append(avg_d)

    print(
        f"Epoch [{epoch + 1}/{FIXED_EPOCHS}] "
        f"Generator Loss: {avg_g:.4f} | "
        f"Discriminator Loss: {avg_d:.4f}"
    )



## 10. Observe the Improved Output

Compare this image grid with the broken/collapsed output above.

**Take a screenshot of the output below for submission.**


In [ ]:

show_generated_images(
    fixed_G,
    "Improved DCGAN - After Label Smoothing",
    "assignment7_outputs/improved_output.png"
)


In [ ]:

plt.figure(figsize=(8, 4))
plt.plot(fixed_g_losses, label="Generator Loss")
plt.plot(fixed_d_losses, label="Discriminator Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Improved DCGAN Training Loss")
plt.legend()
plt.grid(True)
plt.show()



# 11. Side-by-Side Comparison

The same fixed noise vectors are used for both models, making the comparison more consistent.


In [ ]:

broken_G.eval()
fixed_G.eval()

with torch.no_grad():
    broken_images = broken_G(fixed_noise).cpu()
    fixed_images = fixed_G(fixed_noise).cpu()

broken_grid = make_grid(
    broken_images, nrow=8, normalize=True, value_range=(-1, 1)
)
fixed_grid = make_grid(
    fixed_images, nrow=8, normalize=True, value_range=(-1, 1)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(
    broken_grid.permute(1, 2, 0).squeeze(),
    cmap="gray"
)
axes[0].set_title("Broken / Collapsed")
axes[0].axis("off")

axes[1].imshow(
    fixed_grid.permute(1, 2, 0).squeeze(),
    cmap="gray"
)
axes[1].set_title("Improved / Label Smoothing")
axes[1].axis("off")

plt.tight_layout()
plt.show()



# 12. Written Explanation

## 1. What modification caused the model to perform poorly?

The generator was deliberately given much lower capacity by reducing its channel size, and batch normalization was removed from the generator. A significantly higher learning rate was also used. These changes made the adversarial training process unstable and increased the likelihood of mode collapse.

## 2. What did the generated images look like?

The broken model produced images that were very similar to one another. Instead of generating a diverse range of Fashion-MNIST-like samples, the generator tended to focus on a limited type of output. This behavior is characteristic of mode collapse.

## 3. Which technique did you use to improve the model?

Label smoothing was used. Real labels supplied to the discriminator were changed from `1.0` to `0.9`. The retraining also used the more standard DCGAN generator configuration with batch normalization and a lower learning rate.

## 4. Did the solution improve the output? Why?

The improved model should produce more varied and stable outputs than the deliberately unstable model. Label smoothing prevents the discriminator from becoming completely confident about real samples, which can provide the generator with more useful gradients during adversarial training. The more stable generator architecture and lower learning rate also help reduce instability.

> **Important:** The exact degree of mode collapse and improvement depends on the hardware, random seed, dataset, and number of epochs. Your screenshots should be used as the final evidence for what happened in your run.



# 13. Submission Checklist

- [ ] Screenshot of the broken/collapsed output
- [ ] Screenshot of the improved output
- [ ] Short explanation answering all four questions
- [ ] Mention the modification that caused poor performance
- [ ] Mention the improvement technique: label smoothing
- [ ] Compare the diversity of the two generated outputs
